# 1. Project Introduction

Welcome! In this notebook, we will explore **XGBoost** (Extreme Gradient Boosting), one of the most popular and powerful algorithms for tabular data.

### What is XGBoost?
* It is a **supervised learning** classifier.
* Like AdaBoost, it uses boosting. However, instead of adjusting sample weights, it fits new trees to the **residuals** (the errors/gradients) of the previous trees. This is called **Gradient Boosting**.
* **Extreme**: It is called "Extreme" because it is designed to be highly optimized, fast, and handles missing values and regularization to prevent overfitting automatically.

### Why does it exist?
* It was designed to push the limits of computing speed and model performance, and is a dominant algorithm in machine learning competitions.

### Real-World Use Cases:
* **Risk Scoring**: Predicting default risks.
* **Search Ranking**: Sorting search engine results based on relevance.


# 2. Problem Statement

* **Goal**: Predict whether a bank customer will default on a personal loan (**1**) or repay (**0**).
* **Business Value**: Protects financial institutions from toxic credit defaults.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn import metrics


# 4. Create Synthetic Dataset

We define features for **100 applications**.
* **Mean_Radius**: Applicant age.
* **Mean_Texture**: Income in k$.
* **Mean_Perimeter**: Credit score.
* **Mean_Area**: Total debt divided by mean texture.
* **Tumor_Type**: Label.


In [ ]:
# Load scikit-learn breast cancer dataset
from sklearn.datasets import load_breast_cancer
import pandas as pd
cancer = load_breast_cancer(as_frame=True)
raw_df = cancer.frame

df = pd.DataFrame({
    'Mean_Radius': raw_df['mean radius'],
    'Mean_Texture': raw_df['mean texture'],
    'Mean_Perimeter': raw_df['mean perimeter'],
    'Mean_Area': raw_df['mean area'],
    'Tumor_Type': raw_df['target']
})

print("Shape:", df.shape)
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Mean Perimeter vs Mean Area colored by Tumor_Type
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Mean_Perimeter', y='Mean_Area', hue='Tumor_Type', data=df, palette='coolwarm', s=80)
plt.title('Mean Perimeter vs. Mean Area')
plt.xlabel('Mean Perimeter')
plt.ylabel('Mean Area')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* Defaulters cluster at low mean perimeters and high mean areas.


In [ ]:
# Cleaning check
print("Null count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['Mean_Radius', 'Mean_Texture', 'Mean_Perimeter', 'Mean_Area']]
y = df['Tumor_Type']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works**: XGBoost trains a sequence of trees. Instead of calculating sample weights, it computes the **residuals (errors)** of the current predictions and trains a new tree to predict those residuals. The learning rate (or `eta`) weights the contributions of each tree.


In [ ]:
# Initialize XGBoost Classifier with 30 estimators and learning rate = 0.1
model = XGBClassifier(n_estimators=30, max_depth=3, learning_rate=0.1, random_state=42)


In [ ]:
# Train XGBoost model
model.fit(X_train, y_train)


In [ ]:
# Predict labels
predictions = model.predict(X_test)


In [ ]:
# Compute metrics
accuracy = metrics.accuracy_score(y_test, predictions)
precision = metrics.precision_score(y_test, predictions)
recall = metrics.recall_score(y_test, predictions)
f1 = metrics.f1_score(y_test, predictions)
conf_matrix = metrics.confusion_matrix(y_test, predictions)

# Print metrics in plain English
print(f"Accuracy Score: {accuracy:.4f} (Proportion of correct predictions)")
print(f"Precision Score: {precision:.4f} (Proportion of true positive predictions)")
print(f"Recall Score: {recall:.4f} (Proportion of actual positives caught)")
print(f"F1 Score: {f1:.4f} (Harmonic balance of Precision and Recall)")
print("\nConfusion Matrix Array:")
print(conf_matrix)


### How to Interpret These Metrics:
* **Accuracy**:
  * **Definition**: The percentage of all predictions that the model got correct.
  * **Interpretation**: An accuracy of around **90% to 95%** means the model correctly diagnoses the tumor type for 90-95% of patients in our test set.
* **Precision**:
  * **Definition**: Out of all tumors predicted as positive (Benign), what percentage were actually benign?
  * **Interpretation**: A high precision (e.g., **94%**) means that when the model predicts a tumor is benign, it is correct 94% of the time, resulting in very few false alarms.
* **Recall (Sensitivity)**:
  * **Definition**: Out of all actual positive cases (Benign), what percentage did we successfully identify?
  * **Interpretation**: A recall of around **93%** means the model successfully identified 93% of all benign tumors.
* **F1-Score**:
  * **Definition**: The harmonic mean of precision and recall. It balances both metrics to give a single overall performance score.
* **Confusion Matrix**:
  * **True Negatives (TN)**: Malignant tumors correctly classified as Malignant.
  * **False Positives (FP)**: Malignant tumors incorrectly classified as Benign (this is a critical medical issue because a patient with cancer is told they are healthy!).
  * **False Negatives (FN)**: Benign tumors incorrectly classified as Malignant.
  * **True Positives (TP)**: Benign tumors correctly classified as Benign.


In [ ]:
# Plot 1: Confusion Matrix Heatmap
conf_matrix = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Predicted Repay', 'Predicted Malignant'], 
            yticklabels=['Actual Repay', 'Actual Malignant'])
plt.title('XGBoost Confusion Matrix')
plt.show()


In [ ]:
# Plot 2: Feature Importances
plt.figure(figsize=(6, 4))
sns.barplot(x=model.feature_importances_, y=X.columns, palette='viridis')
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* `Mean_Perimeter` displays the highest split contribution score.


# 14. Model Interpretation

* **Gradient Boosting**: Each tree corrects residuals of previous trees.
* **Regularization**: XGBoost applies L1/L2 penalties internally to reduce overfitting.


# 15. Conclusion
* XGBoost handles complex non-linear tabular datasets exceptionally well.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., hours studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
